# Проверка PaddleOCR, MinerU и Unlimited OCR

Ноутбук запускает три независимых backend'а на каждом файле из `inputs/` и собирает время, статус и артефакты.

- **PaddleOCR** — дефолт, работает локально на CPU/GPU.
- **MinerU** — запускается через CLI и возвращает Markdown/JSON.
- **Unlimited OCR** — прямой Transformers inference на NVIDIA GPU.

Тяжёлые зависимости специально не устанавливаются автоматически. Используйте отдельные окружения для production; этот notebook предназначен только для сравнения.

> Положите PDF/JPG/PNG в `inputs/`. Для честного сравнения начните с 1–3 страниц; длинные сканы ограничьте через `MAX_PAGES`. Unlimited OCR требует NVIDIA CUDA; на Mac его ячейка будет корректно пропущена.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Any

INPUTS_DIR = Path("inputs")
OUTPUT_ROOT = Path("benchmark_outputs")
LANG = "ru"  # PaddleOCR: ru, en и т.д.
# Для длинных сканов можно ограничить страницы, например MAX_PAGES = 5
MAX_PAGES: int | None = None

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
INPUT_PATHS = sorted(
    p for p in INPUTS_DIR.iterdir()
    if p.suffix.lower() in {".pdf", ".png", ".jpg", ".jpeg", ".webp", ".tif", ".tiff"}
) if INPUTS_DIR.is_dir() else []
RESULTS: dict[str, dict[str, Any]] = {}
CURRENT_INPUT: Path | None = None
PAGE_IMAGES: list[Path] = []


def safe_stem(path: Path) -> str:
    return "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in path.stem)[:80] or "document"


def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


def record(engine: str, started: float, status: str, **details: Any) -> None:
    file_key = CURRENT_INPUT.name if CURRENT_INPUT is not None else "unknown"
    RESULTS.setdefault(file_key, {})[engine] = {
        "status": status,
        "seconds": round(time.perf_counter() - started, 3),
        **details,
    }


print({
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "inputs_dir": str(INPUTS_DIR),
    "inputs": [p.name for p in INPUT_PATHS],
    "paddleocr_installed": has_module("paddleocr"),
    "mineru_cli": shutil.which("mineru"),
    "torch_installed": has_module("torch"),
})

## Установка (выполняйте только нужные команды)

Рекомендуется отдельное виртуальное окружение для каждого тяжёлого движка.

```bash
# Базовые утилиты notebook
pip install jupyter pillow pymupdf pandas

# PaddleOCR (CPU; сверяйтесь с актуальной инструкцией PaddlePaddle для вашей ОС)
pip install paddlepaddle paddleocr

# MinerU
pip install "mineru[core]"
# CLI должен появиться как: mineru

# Unlimited OCR — только NVIDIA/CUDA; официально протестированные версии на момент создания notebook
pip install torch==2.10.0 torchvision==0.25.0 transformers==4.57.1 \
  Pillow==12.1.1 matplotlib==3.10.8 einops==0.8.2 addict==2.4.0 \
  easydict==1.13 pymupdf==1.27.2.2 psutil==7.2.2
```

На Apple Silicon используйте PaddleOCR и MinerU. Unlimited OCR сейчас ориентирован на NVIDIA CUDA и должен запускаться на отдельной GPU-машине.

In [ ]:
def prepare_page_images(path: Path, dpi: int = 200) -> list[Path]:
    """Возвращает исходное изображение или рендерит PDF постранично."""
    if not path.exists():
        raise FileNotFoundError(f"Укажите существующий INPUT_PATH, сейчас: {path}")
    if path.suffix.lower() != ".pdf":
        return [path]
    if not has_module("fitz"):
        raise RuntimeError("Для PDF установите pymupdf: pip install pymupdf")

    import fitz

    out_dir = OUTPUT_ROOT / safe_stem(path) / "rendered_pages"
    out_dir.mkdir(parents=True, exist_ok=True)
    pages: list[Path] = []
    with fitz.open(path) as document:
        page_count = document.page_count if MAX_PAGES is None else min(document.page_count, MAX_PAGES)
        for index in range(page_count):
            page = document[index]
            scale = min(dpi / 72, 2000 / max(page.rect.width, 1))
            output = out_dir / f"page_{index + 1:04d}.png"
            page.get_pixmap(matrix=fitz.Matrix(scale, scale), alpha=False).save(output)
            pages.append(output)
    return pages


# Пакетный прогон всех файлов из inputs/ — та же логика, что в ячейках ниже.
# Для пошагового разбора одного файла задайте CURRENT_INPUT вручную.
if INPUT_PATHS:
    CURRENT_INPUT = INPUT_PATHS[0]
    PAGE_IMAGES = prepare_page_images(CURRENT_INPUT)
    print(f"{CURRENT_INPUT.name}: подготовлено страниц: {len(PAGE_IMAGES)}")
else:
    raise FileNotFoundError(f"Положите PDF/JPG/PNG в {INPUTS_DIR.resolve()}")
PAGE_IMAGES[:3]

In [ ]:
# PaddleOCR — дефолтный backend
started = time.perf_counter()
try:
    if not has_module("paddleocr"):
        raise RuntimeError("PaddleOCR не установлен; выполните pip install paddlepaddle paddleocr")

    from paddleocr import PaddleOCR

    try:
        # API PaddleOCR 3.x
        paddle = PaddleOCR(
            lang=LANG,
            text_detection_model_name="PP-OCRv5_mobile_det",
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_textline_orientation=False,
        )
        raw_pages = [list(paddle.predict(str(page))) for page in PAGE_IMAGES]
    except TypeError:
        # Совместимость со старыми версиями
        paddle = PaddleOCR(lang=LANG, use_angle_cls=True)
        raw_pages = [paddle.ocr(str(page), cls=True) for page in PAGE_IMAGES]

    def make_jsonable(value: Any) -> Any:
        if isinstance(value, (str, int, float, bool)) or value is None:
            return value
        if isinstance(value, dict):
            return {str(k): make_jsonable(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return [make_jsonable(v) for v in value]
        for attr in ("json", "to_json", "to_dict"):
            candidate = getattr(value, attr, None)
            if candidate is not None:
                candidate = candidate() if callable(candidate) else candidate
                if isinstance(candidate, str):
                    try:
                        candidate = json.loads(candidate)
                    except json.JSONDecodeError:
                        return candidate
                return make_jsonable(candidate)
        return str(value)

    paddle_data = make_jsonable(raw_pages)
    paddle_dir = OUTPUT_ROOT / safe_stem(CURRENT_INPUT) / "paddleocr"
    paddle_dir.mkdir(parents=True, exist_ok=True)
    (paddle_dir / "result.json").write_text(
        json.dumps(paddle_data, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    # Работает с 3.x (rec_texts) и старым форматом [[bbox, (text, score)]].
    texts: list[str] = []
    def collect_text(value: Any) -> None:
        if isinstance(value, dict):
            if isinstance(value.get("rec_texts"), list):
                texts.extend(str(x) for x in value["rec_texts"])
            else:
                for child in value.values():
                    collect_text(child)
        elif isinstance(value, list):
            if len(value) == 2 and isinstance(value[1], (list, tuple)) and value[1] and isinstance(value[1][0], str):
                texts.append(value[1][0])
            else:
                for child in value:
                    collect_text(child)

    collect_text(paddle_data)
    paddle_text = "\n".join(texts)
    (paddle_dir / "result.txt").write_text(paddle_text, encoding="utf-8")
    record("PaddleOCR", started, "ok", pages=len(PAGE_IMAGES), chars=len(paddle_text), output=str(paddle_dir))
    print(paddle_text[:3000])
except Exception as exc:
    record("PaddleOCR", started, "error", error=f"{type(exc).__name__}: {exc}")
    print(RESULTS.get(CURRENT_INPUT.name if CURRENT_INPUT else "unknown", {}).get("PaddleOCR"))

In [ ]:
# MinerU — структурный backend через официальный CLI
MINERU_BACKEND = "pipeline"  # CPU-friendly; для GPU можно выбрать hybrid/vlm по документации
started = time.perf_counter()
try:
    mineru_executable = shutil.which("mineru")
    if not mineru_executable:
        raise RuntimeError("CLI mineru не найден. Установите MinerU в активное окружение.")

    mineru_dir = OUTPUT_ROOT / safe_stem(CURRENT_INPUT) / "mineru"
    mineru_dir.mkdir(parents=True, exist_ok=True)
    command = [
        mineru_executable,
        "-p", str(CURRENT_INPUT.resolve()),
        "-o", str(mineru_dir.resolve()),
        "-b", MINERU_BACKEND,
    ]
    completed = subprocess.run(
        command,
        text=True,
        capture_output=True,
        timeout=3600,
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr[-4000:] or completed.stdout[-4000:])

    markdown_files = sorted(mineru_dir.rglob("*.md"))
    markdown = "\n\n".join(path.read_text(encoding="utf-8") for path in markdown_files)
    record(
        "MinerU",
        started,
        "ok",
        backend=MINERU_BACKEND,
        chars=len(markdown),
        markdown_files=len(markdown_files),
        output=str(mineru_dir),
    )
    print(markdown[:3000] if markdown else f"Готово. Артефакты: {mineru_dir}")
except subprocess.TimeoutExpired:
    record("MinerU", started, "error", error="TimeoutExpired: превышен лимит 3600 сек")
    print(RESULTS.get(CURRENT_INPUT.name if CURRENT_INPUT else "unknown", {}).get("MinerU"))
except Exception as exc:
    record("MinerU", started, "error", error=f"{type(exc).__name__}: {exc}")
    print(RESULTS.get(CURRENT_INPUT.name if CURRENT_INPUT else "unknown", {}).get("MinerU"))

In [ ]:
# Unlimited OCR — long-document backend, официальный Transformers API
# ВНИМАНИЕ: скачивает около 6.7 GB весов и требует NVIDIA CUDA (ориентир: >=12 GB VRAM).
UNLIMITED_MODEL = "baidu/Unlimited-OCR"
UNLIMITED_MAX_PAGES = 40
started = time.perf_counter()
try:
    if not has_module("torch") or not has_module("transformers"):
        raise RuntimeError("Нужны torch и transformers из секции установки")

    import torch
    from transformers import AutoModel, AutoTokenizer

    if not torch.cuda.is_available():
        record("Unlimited OCR", started, "skipped", reason="NVIDIA CUDA недоступна")
        print(RESULTS.get(CURRENT_INPUT.name if CURRENT_INPUT else "unknown", {}).get("Unlimited OCR"))
    else:
        unlimited_dir = OUTPUT_ROOT / safe_stem(CURRENT_INPUT) / "unlimited_ocr"
        unlimited_dir.mkdir(parents=True, exist_ok=True)
        selected_pages = [str(path) for path in PAGE_IMAGES[:UNLIMITED_MAX_PAGES]]

        tokenizer = AutoTokenizer.from_pretrained(UNLIMITED_MODEL, trust_remote_code=True)
        model = AutoModel.from_pretrained(
            UNLIMITED_MODEL,
            trust_remote_code=True,
            use_safetensors=True,
            torch_dtype=torch.bfloat16,
        ).eval().cuda()
        torch.cuda.reset_peak_memory_stats()

        if len(selected_pages) == 1:
            model.infer(
                tokenizer,
                prompt="<image>document parsing.",
                image_file=selected_pages[0],
                output_path=str(unlimited_dir),
                base_size=1024,
                image_size=640,
                crop_mode=True,
                max_length=32768,
                no_repeat_ngram_size=35,
                ngram_window=128,
                save_results=True,
            )
        else:
            model.infer_multi(
                tokenizer,
                prompt="<image>Multi page parsing.",
                image_files=selected_pages,
                output_path=str(unlimited_dir),
                image_size=1024,
                max_length=32768,
                no_repeat_ngram_size=35,
                ngram_window=1024,
                save_results=True,
            )

        text_files = sorted(unlimited_dir.rglob("*.txt")) + sorted(unlimited_dir.rglob("*.md"))
        unlimited_text = "\n\n".join(
            path.read_text(encoding="utf-8", errors="replace") for path in text_files
        )
        record(
            "Unlimited OCR",
            started,
            "ok",
            pages=len(selected_pages),
            chars=len(unlimited_text),
            peak_vram_gb=round(torch.cuda.max_memory_allocated() / 1024**3, 2),
            output=str(unlimited_dir),
        )
        print(unlimited_text[:3000] if unlimited_text else f"Готово. Артефакты: {unlimited_dir}")
        del model
        torch.cuda.empty_cache()
except Exception as exc:
    record("Unlimited OCR", started, "error", error=f"{type(exc).__name__}: {exc}")
    print(RESULTS["Unlimited OCR"])

In [ ]:
# Пакетный прогон всех файлов из inputs/ (та же логика, что в ячейках выше).
# Раскомментируйте, если нужен полный прогон, а не только первый файл.
# from run_ocr_benchmark import main
# main()

# Сводный отчёт
report_path = OUTPUT_ROOT / "benchmark_report.json"
report_path.write_text(json.dumps(RESULTS, ensure_ascii=False, indent=2), encoding="utf-8")

if has_module("pandas"):
    import pandas as pd
    rows = []
    for file_name, engines in RESULTS.items():
        for engine, payload in engines.items():
            rows.append({"file": file_name, "engine": engine, **payload})
    display(pd.DataFrame(rows))
else:
    print(json.dumps(RESULTS, ensure_ascii=False, indent=2))

print(f"Отчёт сохранён: {report_path.resolve()}")

## Как сравнивать результаты

Для каждого файла зафиксируйте:

1. Корректность русского и английского текста.
2. Числа, даты, имена и специальные символы.
3. Порядок чтения колонок.
4. Сохранность таблиц, формул и заголовков.
5. Время, RAM/VRAM и размер артефактов.
6. Повторяемость результата на втором запуске.

Ожидаемое позиционирование:

- **PaddleOCR** — основной быстрый OCR и координаты блоков.
- **MinerU** — лучший кандидат для структурированного Markdown/JSON.
- **Unlimited OCR** — экспериментальный режим длинных документов на GPU.

Не сравнивайте только количество символов: VLM может сформировать хорошо выглядящий, но неверный текст. Критичные числа и сущности проверяйте по оригиналу или размеченному ground truth.